# Spark and Databricks

The previous chapters were about getting the right answer. This chapter is
about getting it efficiently: understanding what a workload actually does on
the cluster, finding where its time and money go, and fixing the expensive
parts. This first notebook looks at what Databricks changes about how Spark
runs, and at the tools it provides for watching workloads. The other two put
them to use:
{doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`
covers the storage layer that most optimizations rely on, and
{doc}`Optimizing Apache Spark <03_optimizing_apache_spark>` the techniques
themselves.

In [1]:
from pyspark.sql import Observation
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

spark.sql("DROP TABLE IF EXISTS workspace.default.monitor01_large_orders");

## What Databricks Changes About Spark

{doc}`What Is Apache Spark? <../01_introduction_to_apache_spark/01_apache_spark_runtime_architecture/01_what_is_apache_spark>`
listed what Databricks adds to open-source Spark: managed compute, the
Databricks Runtime, Delta Lake and Unity Catalog. For performance, four of
these additions matter most:

- **Photon**, the Databricks-native vectorised engine, replaces the
  JVM-based Spark SQL execution engine with a native C++ runtime for the
  operations it supports
  ([What is Photon?](https://docs.databricks.com/aws/en/compute/photon)).
  Plans on Databricks show `Photon...` operators where it is used.
- **Delta Lake** collects per-file statistics on write, which let queries
  skip files that can't match
  ([Data skipping](https://docs.databricks.com/aws/en/delta/data-skipping)),
  and supports clustering and compaction.
- **Serverless compute** takes most tuning decisions out of your hands, as
  shown below.
- **Predictive optimization** automatically runs `OPTIMIZE`, `VACUUM` and
  `ANALYZE` on Unity Catalog managed tables
  ([Predictive optimization](https://docs.databricks.com/aws/en/optimizations/predictive-optimization)),
  as shown in
  {doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`.

On serverless compute, settings that you would tune on a classic cluster are
managed for you: serverless supports setting only a handful of Spark
properties
([Set Spark configuration properties on Databricks](https://docs.databricks.com/aws/en/spark/conf#configure-spark-properties-for-serverless-notebooks-and-jobs)).
The number of shuffle partitions defaults to `auto`, which lets Databricks
determine it
([Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe#enable-auto-optimized-shuffle)):

In [2]:
spark.conf.get("spark.sql.shuffle.partitions")

'auto'

Adaptive Query Execution is enabled by default on Databricks
([Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe#capabilities)),
and its setting is not among the properties serverless lets you change.
Trying to change it fails:

In [3]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

AnalysisException: [CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION] Configuration spark.sql.adaptive.enabled is not available.  SQLSTATE: 42K0I

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at com.databricks.sql.connect.SparkConnectConfig$.assertConfigAllowed(SparkConnectConfig.scala:665)
	at org.apache.spark.sql.connect.service.SparkConnectConfigHandler$RuntimeConfigWrapper.set(SparkConnectConfigHandler.scala:156)
	at org.apache.spark.sql.connect.service.SparkConnectConfigHandler.$anonfun$handleSet$1(SparkConnectConfigHandler.scala:331)
	at org.apache.spark.sql.connect.service.SparkConnectConfigHandler.$anonfun$handleSet$1$adapted(SparkConnectConfigHandler.scala:323)
	at scala.collection.IterableOnceOps.foreach(IterableOnce.scala:630)
	at scala.collection.IterableOnceOps.foreach$(IterableOnce.scala:628)
	at scala.collection.AbstractIterator.foreach(Iterator.scala:1313)
	at org.apache.spark.sql.connect.service.SparkConnectConfigHandler.handleSet(SparkConnectConfigHandler.scala:323)

The error is `CONFIG_NOT_AVAILABLE`: serverless doesn't expose the setting at
all. The consequence for this chapter: on serverless compute, optimizing is less
about Spark configuration and more about how data is laid out and how
queries are written. Those are also the changes that help most on classic
compute.

## Reading a Query Plan

The first monitoring tool needs no UI at all. `explain()` prints the
physical plan Spark will run. Here is a query over `samples.tpch.lineitem`,
a 30-million-row table of order lines from the TPC-H benchmark, that counts
the lines shipped on one day per return flag:

In [4]:
lineitem = spark.read.table("samples.tpch.lineitem")

returns_on_new_year = (
    lineitem
    .filter(F.col("l_shipdate") == "1995-01-01")
    .groupBy("l_returnflag")
    .agg(F.count("*").alias("lines"))
)

returns_on_new_year.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[l_returnflag#33789], functions=[finalmerge_count(merge count#33821L) AS count(1)#33799L])
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#30820]
               +- PhotonShuffleExchangeSink hashpartitioning(l_returnflag#33789, 1024)
                  +- PhotonGroupingAgg(keys=[l_returnflag#33789], functions=[partial_count(1) AS count#33821L])
                     +- PhotonProject [l_returnflag#33789]
                        +- PhotonScan parquet samples.tpch.lineitem[l_returnflag#33789,l_shipdate#33791] DataFilters: [isnotnull(l_shipdate#33791), (l_shipdate#33791 = 1995-01-01)], DictionaryFilters: [(l_shipdate#33791 = 1995-01-01)], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://system-tables-prod-us-east-2-uc-metastore-bucket/metastore/b..., OptionalDataF

:::::{dropdown} Example: Reading the Plan from the Bottom Up
:open:
:color: primary
:icon: pencil
:class-body: example-body

A physical plan is a tree, printed with the first operator to run at the
bottom. Reading the plan above upwards:

**`PhotonScan parquet samples.tpch.lineitem[l_returnflag, l_shipdate]`.**
Only the two columns the query uses are read, and the `DataFilters` show
that the date filter is applied inside the scan.

**`PhotonGroupingAgg(... partial_count(1) ...)`.** Each task counts its own
rows per return flag before anything leaves the executor.

**`PhotonShuffleExchangeSink hashpartitioning(l_returnflag, 1024)`.** The
partial counts are shuffled so that all counts for one flag meet in one
task. The 1024 is the initial number of shuffle partitions; AQE coalesces small
partitions after the shuffle at run time
([Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe#capabilities)).

**`PhotonGroupingAgg(... finalmerge_count ...)`.** The partial counts are
added up into the final result.

The sections after the plan add Databricks-specific information. `Photon
Explanation` says whether every operator runs in Photon, and `Optimizer
Statistics` lists tables without statistics and suggests running
`ANALYZE TABLE`. The whole query has a single shuffle, which is the
minimum for an aggregation.
:::::

`explain(mode="formatted")` splits the output into a plan outline and node
details
([Apache Spark API reference: DataFrame.explain](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html)),
which is easier to read for large plans. How
to interpret the operators is covered in
{doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`.

A plan shows what Spark **intends** to do. To see what it actually did, how
many rows each operator produced, how long it took and where the data was
skewed, you need the query profile or the Spark UI.

## The Query Profile and the Spark UI

Which UI you use depends on the compute:

- **Serverless compute and SQL warehouses** have a **query profile**. Open
  it from the query history, or from the link under the notebook cell that
  ran the query. It shows the executed plan as a graph, with per-operator
  metrics such as time spent, memory peak and rows
  ([Query profile](https://docs.databricks.com/aws/en/sql/user/queries/query-profile)).
  Serverless compute has no Spark UI
  ([Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations#ui-and-logging)).
- **Classic compute** has the **Spark UI**: select the compute on the
  **Compute** page, then the **Spark UI** tab
  ([Debugging with the Spark UI](https://docs.databricks.com/aws/en/compute/troubleshooting/debugging-spark-ui)).
  Its **Jobs** and **Stages** tabs show the jobs, stages and tasks described
  in
  {doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`,
  with summary metrics of the tasks per stage, and the **SQL / DataFrame**
  tab shows executed plans with per-operator metrics
  ([Apache Spark: Web UI](https://spark.apache.org/docs/latest/web-ui.html)).

What to look for is the same in both. Skew shows as a maximum task duration
well above the 75th percentile, and spill as data moved from memory to disk
when Spark runs low on execution memory
([Skew and spill](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/long-spark-stage-page));
Photon doesn't support UDFs
([What is Photon?](https://docs.databricks.com/aws/en/compute/photon)):

| Symptom | Likely cause |
| --- | --- |
| Most time spent in scan operators, many bytes read | The query reads more than it needs: missing filters, no clustering, too many small files |
| Large shuffle reads and writes | Joins or aggregations that move a lot of data; candidates for broadcasting or pre-aggregation |
| One task much slower than the others in a stage | Data skew |
| Spill to disk | Too little memory per task for the data being sorted, joined or aggregated |
| Operators not running in Photon | UDFs or unsupported functions falling back to the Spark engine |

{doc}`Optimizing Apache Spark <03_optimizing_apache_spark>` addresses each
of these.

## Query History in System Tables

Looking at one query at a time does not answer questions like "which of my
queries read the most data last week?". The **system table**
`system.query.history` records queries run on SQL warehouses, serverless
compute for notebooks and jobs, and Lakeflow pipelines
([Query history system table reference](https://docs.databricks.com/aws/en/admin/system-tables/query-history#using-the-query-history-table)).
As a Unity Catalog table, it can be queried like any other; by default only
admins can read it. It has a column for each metric:

In [5]:
history = spark.read.table("system.query.history")

[column for column in history.columns if column.endswith(("_ms", "_bytes", "_files", "_rows"))]

['total_duration_ms',
 'waiting_for_compute_duration_ms',
 'waiting_at_capacity_duration_ms',
 'execution_duration_ms',
 'compilation_duration_ms',
 'total_task_duration_ms',
 'result_fetch_duration_ms',
 'pruned_files',
 'read_files',
 'read_rows',
 'produced_rows',
 'read_bytes',
 'spilled_local_bytes',
 'written_bytes',
 'shuffle_read_bytes',
 'written_rows',
 'written_files',
 'pruned_files_bytes',
 'read_files_bytes']

In the course's workspace, queries sent through Databricks Connect appear
with `client_application` set to `SPARK_CONNECT`, as the filter below
shows. `statement_text` returns `<REDACTED>` unless you are an account admin
or in the `databricks_pii_access` group, and records are typically
available within an hour of a query finishing
([Query history system table reference](https://docs.databricks.com/aws/en/admin/system-tables/query-history#record-availability)),
so very recent queries may be missing. Here are the queries over the TPC-H
sample tables that read the most data in the last week; your output lists
whichever of the course's queries you have run:

In [6]:
(
    history
    .filter(F.col("client_application") == "SPARK_CONNECT")
    .filter(F.col("start_time") >= F.current_timestamp() - F.expr("INTERVAL 7 DAYS"))
    .filter(F.col("statement_text").contains("samples.tpch"))
    .select(
        "statement_type",
        (F.col("total_duration_ms") / 1000).alias("seconds"),
        "read_files",
        "pruned_files",
        F.round(F.col("read_bytes") / 1024**2).alias("read_mb"),
        F.round(F.col("shuffle_read_bytes") / 1024**2).alias("shuffle_mb"),
        "read_rows",
    )
    .orderBy(F.desc("read_mb"))
    .show(10)
)

+--------------+-------+----------+------------+-------+----------+---------+
|statement_type|seconds|read_files|pruned_files|read_mb|shuffle_mb|read_rows|
+--------------+-------+----------+------------+-------+----------+---------+
|        CREATE| 16.936|        20|           0| 1043.0|       0.0| 59999590|
|        CREATE| 12.873|        20|           0| 1043.0|       0.0| 59999590|
|        CREATE|  8.991|        10|           0|  726.0|       0.0| 29999795|
|        CREATE|  10.32|        10|           0|  721.0|       0.0| 29999795|
|        SELECT|   0.87|        10|           0|   46.0|       0.0|    12357|
|        SELECT|  4.006|        71|           0|    3.0|       0.0|        5|
|      DESCRIBE|   0.63|         0|           0|    0.0|       0.0|        0|
|          SHOW|  4.748|         0|           0|    0.0|       0.0|        0|
+--------------+-------+----------+------------+-------+----------+---------+



Two columns deserve a closer look. `read_files` (files read after pruning)
and `pruned_files` show how well a query used **data skipping**
([Query history system table reference](https://docs.databricks.com/aws/en/admin/system-tables/query-history#query-history-system-table-schema)): a query on a well-laid-out table prunes
most files and reads few. `shuffle_mb` shows how much data crossed the
network between stages. Aggregated over all queries, the same table answers
questions like which tables are scanned most often, or which users and jobs
run the most expensive queries.

## Cost in System Tables

The other half of "efficient" is money. Databricks measures usage in
**Databricks Units (DBUs)**, and `system.billing.usage` records billable
usage with columns such as `usage_quantity`, `sku_name`,
`billing_origin_product`, `usage_metadata.job_id` and
`identity_metadata.run_as`
([Billable usage system table reference](https://docs.databricks.com/aws/en/admin/system-tables/billing)). Here is the
usage of this workspace over the last 30 days, by product:

In [7]:
(
    spark.read.table("system.billing.usage")
    .filter(F.col("usage_date") >= F.date_sub(F.current_date(), 30))
    .groupBy("billing_origin_product")
    .agg(F.round(F.sum("usage_quantity"), 2).alias("dbus"))
    .orderBy(F.desc("dbus"))
    .show()
)

+----------------------+----+
|billing_origin_product|dbus|
+----------------------+----+
|                  JOBS|9.40|
|                  APPS|9.00|
|       DEFAULT_STORAGE|8.07|
|                   SQL|5.20|
|           INTERACTIVE|2.40|
|  PREDICTIVE_OPTIMI...|1.38|
|                 GENIE|0.46|
|            NETWORKING|0.00|
+----------------------+----+



`system.billing.list_prices`, a historical log of SKU pricing
([System tables reference](https://docs.databricks.com/aws/en/admin/system-tables/)),
gives the price to turn DBUs into currency, and grouping by
`usage_metadata.job_id` or `identity_metadata.run_as` shows which jobs or
users drive the cost. Before optimizing a workload, check
that it actually matters: a ten-times speed-up of a job that costs a few
cents a month is not worth an afternoon.

Other system tables are useful for monitoring too, for example
`system.lakeflow.job_run_timeline` for job runs, `system.compute` for the
configuration history of classic clusters, and
`system.access.table_lineage` for every read and write of a Unity Catalog
table
([System tables reference](https://docs.databricks.com/aws/en/admin/system-tables/)).

## Observing Metrics in Your Own Code

Sometimes you want metrics about the data itself, such as how many rows a
step wrote or how many had a missing value, for logging or data-quality
checks. Calling `count()` for each one would re-run the whole plan every
time. An **observation** computes such metrics on the data flowing through
that point of the plan, and reports them when the query that runs anyway
completes
([Apache Spark API reference: DataFrame.observe](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.observe.html)):

In [8]:
transactions = spark.read.table("samples.bakehouse.sales_transactions")

write_stats = Observation("large_orders")

large_orders = (
    transactions
    .filter(F.col("quantity") >= 30)
    .observe(
        write_stats,
        F.count(F.lit(1)).alias("rows"),
        F.sum("totalPrice").alias("revenue"),
        F.max("quantity").alias("largest_quantity"),
    )
)

large_orders.write.mode("overwrite").saveAsTable("workspace.default.monitor01_large_orders")

write_stats.get

{'rows': 66, 'revenue': 8352, 'largest_quantity': 60}

The metrics were collected while the table was written, by the same query,
so they cost almost nothing extra. The 66 rows are the same large orders counted
in {doc}`DataFrame API Basics <../02_developing_applications_with_apache_spark/01_dataframe_api_basics>`,
which needed a separate `count()` there. Streaming queries report similar metrics in
every progress report, as shown in
{doc}`Spark Structured Streaming <../03_stream_processing_and_analysis/02_spark_structured_streaming>`.

In [9]:
spark.sql("DROP TABLE IF EXISTS workspace.default.monitor01_large_orders");

## Choosing a Monitoring Tool

| Question | Tool |
| --- | --- |
| What will this query do? | `explain()` |
| What did this query do, operator by operator? | Query profile (serverless, SQL warehouses) or Spark UI (classic compute) |
| Which queries read the most data, or ran longest? | `system.query.history` |
| Where does the money go? | `system.billing.usage` with `system.billing.list_prices` |
| How many rows did my pipeline step write, or how many were invalid? | An `Observation` on the write |

## Common Mistakes

- **Tuning Spark configuration on serverless compute.** Most properties
  can't be set there
  ([Set Spark configuration properties on Databricks](https://docs.databricks.com/aws/en/spark/conf#configure-spark-properties-for-serverless-notebooks-and-jobs)).
  Change how the data is laid out and how the query is written instead.
- **Looking for the Spark UI on serverless.** It isn't available; use the
  query profile
  ([Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations#ui-and-logging)).
- **Expecting a query to show up in `system.query.history` immediately.**
  Records typically take up to an hour
  ([Query history system table reference](https://docs.databricks.com/aws/en/admin/system-tables/query-history#record-availability)).
- **Adding `count()` calls for logging.** Each one runs the plan again; use
  an observation, which reports its metrics when the query completes
  ([Apache Spark API reference: DataFrame.observe](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.observe.html)).
- **Optimizing before measuring cost.** Check `system.billing.usage` first,
  so effort goes to the workloads that cost the most.

Next, {doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`
looks at the storage layer underneath every table in this course.

## Resources

- [Query profile (Databricks)](https://docs.databricks.com/aws/en/sql/user/queries/query-profile)
- [Query history system table reference (Databricks)](https://docs.databricks.com/aws/en/admin/system-tables/query-history)
- [Billable usage system table reference (Databricks)](https://docs.databricks.com/aws/en/admin/system-tables/billing)
- [Debugging with the Spark UI (Databricks)](https://docs.databricks.com/aws/en/compute/troubleshooting/debugging-spark-ui)
- [Web UI (Apache Spark)](https://spark.apache.org/docs/latest/web-ui.html)